In [1]:
import os
os.chdir("..")
%pwd

'/Users/beamaia/Documents/ndb_ufes_data_organizer'

In [2]:
import pathlib as pl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import cv2 as cv

## Get ids

In [3]:
missing_ids = pd.read_csv("missing_ids.csv")
wrong_ids = pd.read_csv("wrong_ids.csv")

In [4]:
missing_ids.head()

,missing_patch_id,origin
0,3,-1
1,10,-1
2,12,-1
3,38,-1
4,39,-1


In [5]:
wrong_ids.head()

,patch,correct_origin
0,p0011,-1.0
1,p0227,18.0
2,p0230,18.0
3,p0281,156.0
4,p0285,156.0


In [6]:
unknown_origin_wrong_ids = wrong_ids[wrong_ids["correct_origin"] == -1]

In [7]:
unknown_origin_wrong_ids.head()

,patch,correct_origin
0,p0011,-1.0
20,p0870,-1.0
21,P0872,-1.0
22,p0873,-1.0
23,p0874,-1.0


In [8]:
len(unknown_origin_wrong_ids)

113

In [9]:
len(missing_ids)

561

In [10]:
missing_ids["patch"] = missing_ids["missing_patch_id"].apply(lambda x: f"p{x:04}" )

In [11]:
missing_ids = missing_ids.drop(columns=["missing_patch_id"])
unknown_origin_wrong_ids = unknown_origin_wrong_ids.reset_index(drop=True)
unknown_origin_wrong_ids["origin"] = -1
unknown_origin_wrong_ids = unknown_origin_wrong_ids.drop(columns=["correct_origin"])

In [12]:
all_missing_ids = pd.concat([missing_ids, unknown_origin_wrong_ids], axis=0)
all_missing_ids["patch"] = all_missing_ids["patch"].apply(lambda x: x.lower())
all_missing_ids = all_missing_ids.sort_values(by=["patch"])
all_missing_ids = all_missing_ids.reset_index(drop=True)

In [13]:
all_missing_ids.head()

,origin,patch
0,-1,p0003
1,-1,p0010
2,-1,p0011
3,-1,p0012
4,-1,p0038


In [14]:
all_missing_ids

,origin,patch
0,-1,p0003
1,-1,p0010
2,-1,p0011
3,-1,p0012
4,-1,p0038
...,...,...
669,-1,p3723
670,-1,p3724
671,-1,p3726
672,-1,p3743


In [15]:
DATA_PATH = pl.Path("data/ndb_ufes/patch/images")
images_paths = sorted([img.name.split(".")[0] for img in DATA_PATH.glob("*.png")])

In [16]:
for row in all_missing_ids.iterrows():
    patch = row[1]["patch"]

    if patch in images_paths:
        print(f"Patch {patch} is present")
        all_missing_ids.at[row[0], "path"] = DATA_PATH / f"{patch}.png"
        

Patch p0003 is present
Patch p0010 is present
Patch p0011 is present
Patch p0012 is present
Patch p0038 is present
Patch p0039 is present
Patch p0040 is present
Patch p0041 is present
Patch p0057 is present
Patch p0126 is present
Patch p0127 is present
Patch p0128 is present
Patch p0129 is present
Patch p0130 is present
Patch p0131 is present
Patch p0132 is present
Patch p0133 is present
Patch p0158 is present
Patch p0159 is present
Patch p0160 is present
Patch p0161 is present
Patch p0162 is present
Patch p0163 is present
Patch p0164 is present
Patch p0313 is present
Patch p0314 is present
Patch p0315 is present
Patch p0316 is present
Patch p0317 is present
Patch p0318 is present
Patch p0319 is present
Patch p0320 is present
Patch p0432 is present
Patch p0433 is present
Patch p0434 is present
Patch p0435 is present
Patch p0436 is present
Patch p0437 is present
Patch p0438 is present
Patch p0439 is present
Patch p0554 is present
Patch p0571 is present
Patch p0638 is present
Patch p0639

In [17]:
all_missing_ids.head()

,origin,patch,path
0,-1,p0003,data/ndb_ufes/patch/images/p0003.png
1,-1,p0010,data/ndb_ufes/patch/images/p0010.png
2,-1,p0011,data/ndb_ufes/patch/images/p0011.png
3,-1,p0012,data/ndb_ufes/patch/images/p0012.png
4,-1,p0038,data/ndb_ufes/patch/images/p0038.png


In [18]:
# all_missing_ids.to_csv("all_missing_ids.csv", index=False)

In [19]:
len(all_missing_ids)

674

# Algorithm

In [210]:
all_missing_ids = pd.read_csv("all_missing_ids.csv")
all_missing_ids = all_missing_ids[all_missing_ids["origin"] == -1]

In [211]:
LINK_PATH = pl.Path("data/ndb_ufes/link")

In [212]:
indexes = range(0, len(all_missing_ids))

In [213]:
patches_name = all_missing_ids["patch"].values
patches_path = all_missing_ids["path"].values

In [214]:
ORIGIN_PATH = pl.Path("data/ndb_ufes/images")

In [215]:
origin_list = sorted([img.name[:-4] for img in ORIGIN_PATH.iterdir()])
origin_paths = sorted([str(img) for img in ORIGIN_PATH.iterdir()])

In [216]:
len(origin_list)

237

In [217]:
index = 18

In [218]:
possible_origins = dict()

img_rgb_og = cv.imread(origin_paths[index])
img_og = cv.cvtColor(img_rgb_og, cv.IMREAD_GRAYSCALE)

for i, (p_img_name, p_img_path) in enumerate(zip(patches_name, patches_path)):
    img = img_og.copy()
    img_rgb = img_rgb_og.copy()

    p_img_rgb = cv.imread(p_img_path)
    p_img = cv.cvtColor(p_img_rgb, cv.IMREAD_GRAYSCALE)
    pw, ph = p_img.shape[:-1]

    print(f"{i} - Seeing if {p_img_name} is from {origin_list[index]}.")

    method = cv.TM_CCOEFF_NORMED

    res = cv.matchTemplate(img_rgb, p_img_rgb, method)
    min_val, max_val, min_loc, max_loc = cv.minMaxLoc(res)
            
    # loc = np.where(res > 0.4)
            
    # if not len(loc[0]) and not len(loc[1]):
    #     continue
            
    top_left = max_loc
    bottom_right = (top_left[0] + pw, top_left[1] + ph)   
            
    cv.rectangle(img_rgb, top_left, bottom_right, 123, 6)

    plt.subplot(1, 2, 1), plt.imshow(p_img_rgb)
    plt.title(f'Patch Point {p_img_name}'), plt.xticks([]), plt.yticks([])
    plt.subplot(1, 2, 2), plt.imshow(img_rgb)
    plt.title(f'Detected point {origin_list[index]}'), plt.xticks([]), plt.yticks([])
    plt.savefig(LINK_PATH / f"temp/{p_img_name}.png")

    plt.close()


0 - Seeing if p0010 is from 0018.
1 - Seeing if p0011 is from 0018.
2 - Seeing if p0012 is from 0018.
3 - Seeing if p0038 is from 0018.
4 - Seeing if p0039 is from 0018.
5 - Seeing if p0040 is from 0018.
6 - Seeing if p0041 is from 0018.
7 - Seeing if p0057 is from 0018.
8 - Seeing if p0126 is from 0018.
9 - Seeing if p0127 is from 0018.
10 - Seeing if p0128 is from 0018.
11 - Seeing if p0129 is from 0018.
12 - Seeing if p0130 is from 0018.
13 - Seeing if p0131 is from 0018.
14 - Seeing if p0132 is from 0018.
15 - Seeing if p0133 is from 0018.
16 - Seeing if p0158 is from 0018.
17 - Seeing if p0159 is from 0018.
18 - Seeing if p0160 is from 0018.
19 - Seeing if p0161 is from 0018.
20 - Seeing if p0162 is from 0018.
21 - Seeing if p0163 is from 0018.
22 - Seeing if p0164 is from 0018.
23 - Seeing if p0313 is from 0018.
24 - Seeing if p0314 is from 0018.
25 - Seeing if p0315 is from 0018.
26 - Seeing if p0316 is from 0018.
27 - Seeing if p0317 is from 0018.
28 - Seeing if p0318 is from 0

# Algoritmo 2

In [ ]:
all_missing_ids = pd.read_csv("all_missing_ids.csv")
all_missing_ids = all_missing_ids[all_missing_ids["origin"] == -1]

In [219]:
possible_origins = dict()

img_rgb_og = cv.imread(origin_paths[index])
img_og = cv.cvtColor(img_rgb_og, cv.IMREAD_GRAYSCALE)

for i, (p_img_name, p_img_path) in enumerate(zip(patches_name, patches_path)):
    img = img_og.copy()
    img_rgb = img_rgb_og.copy()

    p_img_rgb = cv.imread(p_img_path)
    p_img = cv.cvtColor(p_img_rgb, cv.IMREAD_GRAYSCALE)
    pw, ph = p_img.shape[:-1]

    for img_name, img_part in zip(origin_list, origin_paths):
        img_rgb = cv.imread(img_part)
        img = cv.cvtColor(img_rgb, cv.IMREAD_GRAYSCALE)

        print(f"{i} - Seeing if {p_img_name} is from {img_name}.")

        method = cv.TM_CCOEFF_NORMED

        res = cv.matchTemplate(img_rgb, p_img_rgb, method)
        min_val, max_val, min_loc, max_loc = cv.minMaxLoc(res)
                
        # loc = np.where(res > 0.4)
                
        # if not len(loc[0]) and not len(loc[1]):
        #     continue
                
        top_left = max_loc
        bottom_right = (top_left[0] + pw, top_left[1] + ph)   
                
        cv.rectangle(img_rgb, top_left, bottom_right, 123, 6)

        plt.subplot(1, 2, 1), plt.imshow(p_img_rgb)
        plt.title(f'Patch Point {p_img_name}'), plt.xticks([]), plt.yticks([])
        plt.subplot(1, 2, 2), plt.imshow(img_rgb)
        plt.title(f'Detected point {img_name}'), plt.xticks([]), plt.yticks([])
        plt.savefig(LINK_PATH / f"temp/patch_{p_img_name}_origin_{img_name}.png")

        plt.close()

    

0 - Seeing if p0010 is from 0000.
0 - Seeing if p0010 is from 0001.
0 - Seeing if p0010 is from 0002.
0 - Seeing if p0010 is from 0003.
0 - Seeing if p0010 is from 0004.
0 - Seeing if p0010 is from 0005.
0 - Seeing if p0010 is from 0006.
0 - Seeing if p0010 is from 0007.
0 - Seeing if p0010 is from 0008.
0 - Seeing if p0010 is from 0009.
0 - Seeing if p0010 is from 0010.
0 - Seeing if p0010 is from 0011.
0 - Seeing if p0010 is from 0012.
0 - Seeing if p0010 is from 0013.
0 - Seeing if p0010 is from 0014.
0 - Seeing if p0010 is from 0015.
0 - Seeing if p0010 is from 0016.
0 - Seeing if p0010 is from 0017.
0 - Seeing if p0010 is from 0018.
0 - Seeing if p0010 is from 0019.
0 - Seeing if p0010 is from 0020.
0 - Seeing if p0010 is from 0021.
0 - Seeing if p0010 is from 0022.
0 - Seeing if p0010 is from 0023.
0 - Seeing if p0010 is from 0024.
0 - Seeing if p0010 is from 0025.
0 - Seeing if p0010 is from 0026.
0 - Seeing if p0010 is from 0027.
0 - Seeing if p0010 is from 0028.
0 - Seeing if 

: 